# Remake Plots

## Objective
Remake heatmaps and volcano plots with alternate thresholds for follow-up analyses (logFC = 0 by default).

Uses comparison CSV files from `workflow/04_secondary_analyses/CSV/` and sample metadata from `workflow/00_raw_data/config/sample_metadata.csv`.


# Import libraries

In [1]:
import sys
import subprocess
import glob
import pandas as pd

# Add python scripts to import path

In [2]:
sys.path.append('../python')

# Import Scripts

In [6]:
from helpers import get_columns_for_group, load_comparison_files
from volcano_plotter import plot_volcano
from heatmap_plotter import plot_heatmap

# Load metadata

In [7]:
metadata = pd.read_csv('../../00_raw_data/config/sample_metadata.csv')

# Create Volcano plots and Heatmaps

In [ ]:
for file, group1, group2 in load_comparison_files('../../04_secondary_analyses/CSV/'):
    df = pd.read_csv(file, index_col=0)
    print(group1, "vs", group2, df.shape)
    comparison = (group1, group2)
    
    # Retrieve sample names for both groups
    samples_group1 = get_columns_for_group(metadata, treatmentGroupColumnID='group', group_name=group1)
    samples_group2 = get_columns_for_group(metadata, treatmentGroupColumnID='group', group_name=group2)
    
  
    ## Create Plots 
    ### Volcano Plot
    hoverDataList1 = ['Feature', 'Description', 'FoldChange'] 

    ###### create a title for each comparison
    pTitle = " vs ".join(comparison)

    ###### create an outfile name for each comparison
    outTitle = "-vs-".join(comparison)
    outFilename = "volcano_plot_" + outTitle
    plot_volcano(df=df,columns=['p-value_StudentTtest','log2FoldChange'],pvalue_limit=0.05, fold_change_limit=0, hoverDataList=hoverDataList1, plotTitle=pTitle, outFileName=outFilename)
    plot_volcano(df=df,columns=['q-value_StudentTtest','log2FoldChange'],pvalue_limit=0.05, fold_change_limit=0, hoverDataList=hoverDataList1, plotTitle=pTitle, outFileName=outFilename, q_values=True)

    print(samples_group1)
    print(samples_group2)
    plot_heatmap(df=df,columns=['p-value_StudentTtest','log2FoldChange'], pvalueLimit=0.05, foldChangeLimit=0, comparison=comparison, samples_group1=samples_group1, samples_group2=samples_group2)
    plot_heatmap(df=df,columns=['q-value_StudentTtest','log2FoldChange'], pvalueLimit=0.05, foldChangeLimit=0, comparison=comparison, samples_group1=samples_group1, samples_group2=samples_group2, q_values=True)



# Organize the output files

In [11]:
# Make dirs and move output files
commands = [
    'mkdir -p ../../04_secondary_analyses/PNG',
    'mkdir -p ../../04_secondary_analyses/SVG',
    'mkdir -p ../../04_secondary_analyses/HTML',
    'mv ./*.png ../../04_secondary_analyses/PNG',
    'mv ./*.svg ../../04_secondary_analyses/SVG',
    'mv ./*.html ../../04_secondary_analyses/HTML'
]

for command in commands:
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Command {command} executed successfully.")
    else:
        print(f"Error executing command {command}: {result.stderr}")

Command mkdir -p PNG executed successfully.
Command mkdir -p SVG executed successfully.
Command mkdir -p HTML executed successfully.
Command mv ./*.png PNG executed successfully.
Command mv ./*.svg SVG executed successfully.
Command mv ./*.html HTML executed successfully.


## Record the versions of python and python libraries used

In [16]:
# Print Python version
print("Python version:", sys.version)

# Print library versions
print("pandas version:", pd.__version__)


Python version: 3.10.2 | packaged by conda-forge | (main, Jan 14 2022, 08:02:09) [GCC 9.4.0]
pandas version: 2.2.3


In [ ]:
files_path = "workflow/04_secondary_analyses/"

### Interactive Plots

print(f"""## Interactive Volcano Plots and Heatmaps

These plots can be found in the following folder:

* [{files_path}HTML]({files_path}HTML)
""")

### Static plots
print(f"""## Static Vocano plots, and Heatmaps

These plots can be found in the following folder:

* [{files_path}PNG]({files_path}PNG)

Volcano plots and Heatmaps in Scalable Vector Graphics format can be found in the following folder:
* [{files_path}SVG]({files_path}SVG)
""")


def dis_plot(files):
    for file in files:
        rel = Path(file).name
        print("###", rel)
        print("")
        if rel.startswith('volcano'):
            print(f"![]({files_path}PNG/{rel})")
        elif rel.startswith('Heatmap'):
            print(f"![]({files_path}PNG/{rel})")
        else:
            print(f"![]({files_path}{rel})")
        print("")

### Volcano Plots
print("""
## Volcano Plots 
Here are the volcano plots for the comparisons. These plots show the significance of the proteins on the y-axis and the fold change on the x-axis. The red and blue points are the proteins that are significantly differentially abundant in the positive and negative direction (adjusted p-value < 0.05).

""")
files = glob.glob("../../04_secondary_analyses/PNG/volcano*P.png")
print("## Volcano plots using Pvalues")
print("")
print("These plots show the significant proteins using a pvalue cutoff of p<0.05 and a logFC of 0")
print("")
dis_plot(files)
    
files = glob.glob("../../04_secondary_analyses/PNG/volcano_*Q.png")
print("## Volcano plots using Qvalues")
print("")
print("These plots show the significant proteins using a qvalue cutoff of q<0.05 and a logFC of 0")
print("")
dis_plot(files)

### Heatmaps
print("""
## Heatmaps
Here are the heatmaps for the comparisons. These plots show the significant proteins on the y-axis and the sample name on the x-axis. The color of heatmap is log2 of abundance values of the proteins.

""")
files = glob.glob("../../04_secondary_analyses/PNG/Heatmap*P.png")
print("## Heatmaps using Pvalues")
print("")
print("These plots show the significant proteins using a pvalue cutoff of p<0.05 and a logFC of 0")
print("")
dis_plot(files)
files = glob.glob("../../04_secondary_analyses/PNG/Heatmap_*Q.png")
print("## Heatmaps using Qvalues")
print("")
print("These plots show the significant proteins using a qvalue cutoff of q<0.05 and a logFC of 0")
print("")
dis_plot(files)

